<a href="https://colab.research.google.com/github/Sai-Srinivas7/TextGuard-2.0/blob/main/NLP_Pipeline_GoogleStyle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# End-to-End NLP Pipeline: Governance & Compliance

This consolidated section contains the complete, cleaned-up pipeline for processing project documents, training a risk classification model, and annotating PDFs.

## 1a. Setup and Installation
Install required libraries and models.

In [ ]:
# Install required libraries and download the spaCy language model.
!pip install -q python-docx spacy transformers datasets evaluate accelerate PyMuPDF
!python -m spacy download en_core_web_sm -q

## 1b. Reproducibility Seed
Set a global seed across Python, NumPy, and PyTorch (CPU + CUDA) so model training and data splits are reproducible across runs.

In [ ]:
import random
import numpy as np
import torch
# -- Constant --
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## 2. Data Loading & Preprocessing
Load the dataset, clean the text, and map labels.


In [ ]:
import re

import pandas as pd

# -- Constants --
DATA_PATH = 'Labeled_docs_data.csv'
LABEL_MAP = {'no_risk': 0, 'risk': 1}


def clean_text(text: str) -> str:
    """Normalizes a raw text string for model input.

    Lowercases the text, strips non-alphabetic characters, and collapses
    whitespace to a single space.

    Args:
        text: The raw input string to clean.

    Returns:
        A normalized string, or an empty string if the input is not a str.
    """
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Load dataset and drop rows with missing chunk text.
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['chunk_text']).copy()

# Apply text cleaning and encode labels numerically.
df['cleaned_text'] = df['chunk_text'].apply(clean_text)
df['label'] = df['risk_label'].map(LABEL_MAP)

df_clean = df.dropna(subset=['cleaned_text', 'label']).copy()
df_clean['label'] = df_clean['label'].astype(int)

display(df_clean[['chunk_text', 'cleaned_text', 'label']].head(3))

## 3. Exploratory NLP (NER & Topic Modeling)
Extract entities using spaCy and find latent topics using LDA.

In [ ]:
import spacy
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

# -- Constants --
N_TOPICS = 5
TOP_N_WORDS = 10

# Named Entity Recognition on a sample document.
nlp = spacy.load('en_core_web_sm')
print('Sample Entities from first chunk:')
sample_doc = nlp(str(df_clean['chunk_text'].iloc[0]))
print([(ent.text, ent.label_) for ent in sample_doc.ents])

# Topic Modeling with Latent Dirichlet Allocation.
print('\nExtracting Top Topics...')
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')
dtm = vectorizer.fit_transform(df_clean['cleaned_text'])

lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS, random_state=42
)
lda_model.fit(dtm)

feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(lda_model.components_):
    top_words = ' '.join(
        feature_names[i]
        for i in topic.argsort()[: -TOP_N_WORDS - 1 : -1]
    )
    print(f'Topic {topic_idx}: {top_words}')

## 4. Advanced Risk Classification Model Training
Fine-tune a DistilBERT model using **Document-Level Splits** (GroupShuffleSplit) to prevent data leakage, and a custom **Focal Loss** implementation to handle class imbalance.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import evaluate
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

# -- Constants --
MODEL_NAME = 'distilbert-base-uncased'
MAX_SEQ_LEN = 128
TEST_SIZE = 0.2


# 1. Document-level splits to prevent data leakage.
# Extract a document_id from chunk_id, e.g.:
#   'Project_Document_Disclosure_32681_secI_chunk0' -> first 4 underscore-
#   separated parts.
df_clean['document_id'] = df_clean['chunk_id'].apply(
    lambda x: '_'.join(str(x).split('_')[:4])
)

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=42)
train_idx, test_idx = next(
    gss.split(df_clean, groups=df_clean['document_id'])
)

train_df = df_clean.iloc[train_idx]
test_df = df_clean.iloc[test_idx]

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
print(f'Training chunks: {len(train_df)} | Testing chunks: {len(test_df)}')

# 2. Tokenization.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_function(examples: dict) -> dict:
    """Tokenizes cleaned text with padding and truncation.

    Args:
        examples: A batch dict containing a 'cleaned_text' key.

    Returns:
        A dict of tokenized inputs (input_ids, attention_mask, etc.).
    """
    return tokenizer(
        examples['cleaned_text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )


tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)


# 3. Custom Trainer that substitutes Focal Loss for standard cross-entropy.
class FocalLossTrainer(Trainer):
    """HuggingFace Trainer that applies Focal Loss to handle class imbalance.

    Attributes:
        alpha: Scalar weighting factor that up-weights the minority class.
        gamma: Focusing exponent that down-weights easy examples.
    """

    def __init__(
        self,
        *args,
        alpha: float = 0.25,
        gamma: float = 2.0,
        **kwargs,
    ) -> None:
        """Initializes FocalLossTrainer with Focal Loss hyperparameters.

        Args:
            *args: Positional arguments forwarded to the base Trainer.
            alpha: Focal loss alpha weighting factor. Defaults to 0.25.
            gamma: Focal loss gamma focusing parameter. Defaults to 2.0.
            **kwargs: Keyword arguments forwarded to the base Trainer.
        """
        super().__init__(*args, **kwargs)
        self.alpha = alpha
        self.gamma = gamma

    def compute_loss(
        self,
        model,
        inputs: dict,
        return_outputs: bool = False,
        **kwargs,
    ):
        """Computes Focal Loss for the current batch.

        Args:
            model: The model being trained.
            inputs: A batch of model inputs, including 'labels'.
            return_outputs: If True, returns (loss, outputs) instead of loss.
            **kwargs: Additional keyword arguments.

        Returns:
            The scalar focal loss tensor, or a (loss, outputs) tuple when
            return_outputs is True.
        """
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits

        ce_loss = nn.functional.cross_entropy(
            logits, labels, reduction='none'
        )
        pt = torch.exp(-ce_loss)
        focal_loss = (self.alpha * (1 - pt) ** self.gamma * ce_loss).mean()
        return (focal_loss, outputs) if return_outputs else focal_loss


# 4. Evaluation metric.
accuracy_metric = evaluate.load('accuracy')


def compute_metrics(eval_pred) -> dict:
    """Computes classification accuracy from logits and ground-truth labels.

    Args:
        eval_pred: A named tuple with (logits, labels) fields.

    Returns:
        A dict mapping 'accuracy' to its scalar value.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)


# 5. Model initialization, configuration, and training.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

training_args = TrainingArguments(
    output_dir='./risk_model_advanced',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy='epoch',
)

trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print('\nTraining advanced model with Focal Loss...')
trainer.train()
print('\nEvaluation Results:', trainer.evaluate())

## 4b. Hierarchical Attention Network (HAN) Architecture
Define a custom PyTorch model that processes chunks using DistilBERT and aggregates them into a document-level prediction using an Attention mechanism.

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

# -- Constant (reused from cell above) --
MODEL_NAME = 'distilbert-base-uncased'


class DocumentAttentionLayer(nn.Module):
    """Soft-attention layer that aggregates chunk embeddings into a document vector.

    Attributes:
        attention: A two-layer MLP that scores each chunk embedding.
    """

    def __init__(self, hidden_size: int) -> None:
        """Initializes the attention mechanism.

        Args:
            hidden_size: Dimensionality of the input chunk embeddings.
        """
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1, bias=False),
        )

    def forward(
        self, chunk_embeddings: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Computes a weighted document embedding from chunk representations.

        Args:
            chunk_embeddings: Float tensor of shape
                (batch_size, num_chunks, hidden_size).

        Returns:
            A tuple (doc_embedding, attn_weights) where doc_embedding has
            shape (batch_size, hidden_size) and attn_weights has shape
            (batch_size, num_chunks, 1).
        """
        # Score each chunk: (batch_size, num_chunks, 1).
        attn_scores = self.attention(chunk_embeddings)
        attn_weights = torch.softmax(attn_scores, dim=1)

        # Weighted sum across the chunk dimension.
        doc_embedding = torch.sum(attn_weights * chunk_embeddings, dim=1)
        return doc_embedding, attn_weights


class HierarchicalAttentionNetwork(nn.Module):
    """Two-level encoder: DistilBERT for chunks, soft-attention for documents.

    Architecture:
        1. Chunk encoder: DistilBERT encodes each chunk independently.
        2. Document attention: soft-attention pools chunk vectors.
        3. Classifier: linear layer over the document embedding.

    Attributes:
        chunk_encoder: Pre-trained DistilBERT model.
        doc_attention: DocumentAttentionLayer instance.
        classifier: Linear classification head.
    """

    def __init__(
        self,
        model_name: str = MODEL_NAME,
        num_labels: int = 2,
    ) -> None:
        """Initializes the HAN with a DistilBERT backbone.

        Args:
            model_name: HuggingFace model identifier for the chunk encoder.
            num_labels: Number of output classes. Defaults to 2 (binary).
        """
        super().__init__()
        self.chunk_encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.chunk_encoder.config.hidden_size
        self.doc_attention = DocumentAttentionLayer(hidden_size)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Runs a forward pass over a batch of documents.

        Args:
            input_ids: LongTensor of shape (batch_size, num_chunks, seq_len).
            attention_mask: Binary tensor matching input_ids shape.

        Returns:
            A tuple (logits, attn_weights) where logits has shape
            (batch_size, num_labels) and attn_weights has shape
            (batch_size, num_chunks, 1).
        """
        batch_size, num_chunks, seq_len = input_ids.size()

        # Flatten to (batch_size * num_chunks, seq_len) for DistilBERT.
        input_ids_flat = input_ids.view(-1, seq_len)
        attention_mask_flat = attention_mask.view(-1, seq_len)

        # Use the [CLS] token (index 0) as the chunk representation.
        outputs = self.chunk_encoder(
            input_ids=input_ids_flat,
            attention_mask=attention_mask_flat,
        )
        chunk_embeddings = outputs.last_hidden_state[:, 0, :]

        # Reshape to (batch_size, num_chunks, hidden_size).
        chunk_embeddings = chunk_embeddings.view(batch_size, num_chunks, -1)

        doc_embedding, attn_weights = self.doc_attention(chunk_embeddings)
        logits = self.classifier(doc_embedding)
        return logits, attn_weights


han_model = HierarchicalAttentionNetwork()
print('HAN Architecture successfully defined!')

### 4c. HAN Data Preparation
To train the HAN model, we need to convert our flat chunk-level dataset into a 3D tensor: `(batch_size, num_chunks, sequence_length)`. We'll group the chunks by `document_id` and create a custom PyTorch `Dataset`.

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset

# -- Constants --
MAX_CHUNKS = 10
MAX_SEQ_LEN = 128
# Small batch: 3-D tensors consume significantly more VRAM than 2-D ones.
BATCH_SIZE = 4


class HANDataset(Dataset):
    """PyTorch Dataset that groups text chunks into document-level tensors.

    Each item is a 3-D tensor of shape (num_chunks, seq_len) representing
    one document, enabling the HAN to compute document-level predictions.

    Attributes:
        doc_ids: Array of unique document identifiers.
        df: Source DataFrame containing chunk text and labels.
        tokenizer: HuggingFace tokenizer used for encoding chunks.
        max_chunks: Maximum chunks per document (padded/truncated).
        max_seq_len: Maximum token length per chunk.
    """

    def __init__(
        self,
        df,
        tokenizer,
        max_chunks: int = MAX_CHUNKS,
        max_seq_len: int = MAX_SEQ_LEN,
    ) -> None:
        """Initializes the dataset by indexing unique document IDs.

        Args:
            df: DataFrame with 'document_id', 'cleaned_text', and 'label'.
            tokenizer: HuggingFace tokenizer for encoding chunk text.
            max_chunks: Max chunks retained per document. Defaults to 10.
            max_seq_len: Max token length per chunk. Defaults to 128.
        """
        self.doc_ids = df['document_id'].unique()
        self.df = df
        self.tokenizer = tokenizer
        self.max_chunks = max_chunks
        self.max_seq_len = max_seq_len

    def __len__(self) -> int:
        """Returns the number of documents in the dataset."""
        return len(self.doc_ids)

    def __getitem__(self, idx: int) -> dict:
        """Returns tokenized tensors for the document at position idx.

        The document-level label is 1 (risk) if any constituent chunk
        carries a risk label, and 0 (no risk) otherwise.

        Args:
            idx: Integer index into the unique document ID list.

        Returns:
            A dict with keys:
                - 'input_ids': LongTensor (max_chunks, max_seq_len).
                - 'attention_mask': LongTensor (max_chunks, max_seq_len).
                - 'labels': Scalar LongTensor with the document risk label.
        """
        doc_id = self.doc_ids[idx]
        doc_data = self.df[self.df['document_id'] == doc_id]
        doc_chunks = doc_data['cleaned_text'].tolist()

        # Mark document as risky if any of its chunks are risky.
        doc_label = int(doc_data['label'].max())

        # Truncate or pad with empty strings to a fixed chunk count.
        chunks = doc_chunks[: self.max_chunks]
        chunks += [''] * (self.max_chunks - len(chunks))

        encodings = self.tokenizer(
            chunks,
            padding='max_length',
            truncation=True,
            max_length=self.max_seq_len,
            return_tensors='pt',
        )

        return {
            'input_ids': encodings['input_ids'],
            'attention_mask': encodings['attention_mask'],
            'labels': torch.tensor(doc_label, dtype=torch.long),
        }


print('Creating HAN DataLoaders...')
train_han_dataset = HANDataset(train_df, tokenizer)
test_han_dataset = HANDataset(test_df, tokenizer)

train_han_loader = DataLoader(
    train_han_dataset, batch_size=BATCH_SIZE, shuffle=True
)
test_han_loader = DataLoader(test_han_dataset, batch_size=BATCH_SIZE)
print(
    f'Train documents: {len(train_han_dataset)} | '
    f'Test documents: {len(test_han_dataset)}'
)

### 4d. HAN Training Loop
We write a standard PyTorch training loop to process the 3D batches and update the model weights.

In [ ]:
import torch.optim as optim
from tqdm.auto import tqdm

# -- Constants --
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
han_model.to(device)

optimizer = optim.AdamW(han_model.parameters(), lr=LEARNING_RATE)
loss_fn = torch.nn.CrossEntropyLoss()

print(f'Starting HAN training on {device}...')

for epoch in range(NUM_EPOCHS):
    han_model.train()
    total_loss = 0.0

    progress_bar = tqdm(
        train_han_loader, desc=f'Epoch {epoch + 1}/{NUM_EPOCHS}'
    )
    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits, _ = han_model(input_ids, attention_mask)
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})

    avg_loss = total_loss / len(train_han_loader)
    print(f'Epoch {epoch + 1} Completed | Average Loss: {avg_loss:.4f}')

print('HAN Training Complete!')

### 4e. HAN Evaluation
Let's evaluate our trained document-level HAN model on the test dataset.

In [ ]:
import torch
from sklearn.metrics import accuracy_score, classification_report

han_model.eval()
all_preds: list = []
all_labels: list = []

print('Evaluating HAN on test set...')
with torch.no_grad():
    for batch in test_han_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits, _ = han_model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f'Document-Level Test Accuracy: {acc * 100:.2f}%')
print('\nClassification Report:')
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=['no_risk', 'risk'],
        zero_division=0,
    )
)

## 5. PDF Spatial Annotation
Use the trained model to find and highlight high-risk sentences in a PDF document.

In [ ]:
import os

import fitz  # PyMuPDF
import torch
import torch.nn.functional as F

# -- Constants --
PDF_FILENAME = 'TC Abstract - RG-T4682.pdf'
PDF_PATH = f'{PDF_FILENAME}'
CONFIDENCE_THRESHOLD = 0.75
MIN_SENTENCE_LEN = 30

if not os.path.exists(PDF_PATH):
    print(f'File not found: {PDF_PATH}')
else:
    doc = fitz.open(PDF_PATH)
    risk_count = 0

    for page in doc:
        for block in page.get_text('blocks'):
            if block[6] != 0:  # Skip non-text blocks.
                continue

            block_text = block[4].strip()
            spacy_doc = nlp(block_text)

            for sent in spacy_doc.sents:
                sentence_text = sent.text.strip()
                if len(sentence_text) <= MIN_SENTENCE_LEN:
                    continue

                inputs = tokenizer(
                    sentence_text,
                    return_tensors='pt',
                    padding=True,
                    truncation=True,
                    max_length=MAX_SEQ_LEN,
                )
                inputs = {
                    k: v.to(trainer.model.device)
                    for k, v in inputs.items()
                }

                with torch.no_grad():
                    outputs = trainer.model(**inputs)
                    probs = F.softmax(outputs.logits, dim=-1)
                    risk_prob = probs[0][1].item()

                if risk_prob > CONFIDENCE_THRESHOLD:
                    risk_count += 1
                    for inst in page.search_for(sentence_text):
                        annot = page.add_rect_annot(inst)
                        annot.set_colors(stroke=(1, 0, 0))  # Red highlight.
                        annot.update()

    output_pdf_path = f'Annotated_Clean_{PDF_FILENAME}'
    doc.save(output_pdf_path)
    print(
        f'Highlighted {risk_count} risk sentences with '
        f'>{CONFIDENCE_THRESHOLD * 100:.0f}% confidence.'
    )
    print(f'Saved annotated PDF to: {output_pdf_path}')

## 5b. HAN PDF Inference & Attention Extraction
Let's apply our custom Hierarchical Attention Network to a PDF. Because HAN uses a Document-Level Attention mechanism, it can not only predict if the document is risky but also tell us which specific chunks it focused on to make that decision!

In [ ]:
import os

import fitz  # PyMuPDF
import torch
import torch.nn.functional as F

# -- Constants --
PDF_FILENAME = 'TC Abstract - RG-T4682.pdf'
PDF_PATH = f'{PDF_FILENAME}'
MAX_CHUNKS = 10
MAX_SEQ_LEN = 128
MIN_CHUNK_WORDS = 5

print(f'Processing PDF: {PDF_FILENAME} for HAN prediction...')

# 1. Extract valid text chunks from the PDF.
chunks: list = []
if os.path.exists(PDF_PATH):
    doc = fitz.open(PDF_PATH)
    for page in doc:
        for block in page.get_text('blocks'):
            if block[6] != 0:
                continue
            text = block[4].strip().replace('\n', ' ')
            if len(text.split()) > MIN_CHUNK_WORDS:
                chunks.append(text)
else:
    print(f'File not found: {PDF_PATH}')

print(f'Extracted {len(chunks)} valid text chunks from the PDF.')

# 2. Build a padded, fixed-length chunk sequence for HAN input.
doc_chunks = chunks[:MAX_CHUNKS]
doc_chunks += [''] * (MAX_CHUNKS - len(doc_chunks))

encodings = tokenizer(
    doc_chunks,
    padding='max_length',
    truncation=True,
    max_length=MAX_SEQ_LEN,
    return_tensors='pt',
)

# Add batch dimension: (1, num_chunks, seq_len).
input_ids = encodings['input_ids'].unsqueeze(0).to(device)
attention_mask = encodings['attention_mask'].unsqueeze(0).to(device)

# 3. Run document-level inference.
label_map = {0: 'No Risk', 1: 'Risk'}
han_model.eval()
with torch.no_grad():
    logits, attn_weights = han_model(input_ids, attention_mask)
    probs = F.softmax(logits, dim=1)
    pred_class = torch.argmax(logits, dim=1).item()
    risk_prob = probs[0][1].item()

print('\n' + '=' * 40)
print('       HAN PREDICTION RESULTS')
print('=' * 40)
print(f'Predicted Document Label : {label_map[pred_class]}')
print(f'Confidence (Risk Prob.)  : {risk_prob * 100:.2f}%')
print('=' * 40)

# 4. Surface the chunk the model attended to most.
# attn_weights shape: (1, num_chunks, 1) -> flatten to a 1-D array.
attn_weights_flat = attn_weights.squeeze().cpu().numpy()
top_chunk_idx = int(attn_weights_flat.argmax())
top_score = attn_weights_flat[top_chunk_idx]

if doc_chunks[top_chunk_idx].strip():
    print(f'\nMost Critical Chunk (Attention Weight: {top_score:.4f}):')
    print(f'"{doc_chunks[top_chunk_idx]}"')
else:
    print(
        '\nThe model attended mostly to padding, indicating no strong '
        'risk signals in the sampled chunks.'
    )

## 6. Extension: Multi-Label Risk Categorization
To predict multiple specific risk categories simultaneously (Financial, Social, Environmental), we map the labels and update our model architecture to use `BCEWithLogitsLoss`.

In [ ]:
_FINANCIAL_KEYWORDS = frozenset(
    ['budget', 'cost', 'finance', 'financial', 'economic', 'funding']
)
_SOCIAL_KEYWORDS = frozenset(
    ['community', 'people', 'social', 'indigenous', 'human', 'health', 'livelihood']
)
_ENVIRONMENTAL_KEYWORDS = frozenset(
    ['environment', 'nature', 'water', 'forest', 'climate', 'pollution', 'biodiversity']
)


def assign_risk_categories(text: str) -> tuple:
    """Assigns binary risk flags via keyword matching.

    Uses heuristic keyword matching to bootstrap multi-label annotations
    for Financial, Social, and Environmental risk categories.

    Note:
        These labels are approximations for demonstration purposes. A
        production system requires manually validated ground truth.

    Args:
        text: The cleaned document chunk text.

    Returns:
        A tuple (financial, social, environmental) where each element is
        1 if corresponding risk keywords are present, 0 otherwise.
    """
    text = str(text).lower()
    financial = int(any(w in text for w in _FINANCIAL_KEYWORDS))
    social = int(any(w in text for w in _SOCIAL_KEYWORDS))
    environmental = int(any(w in text for w in _ENVIRONMENTAL_KEYWORDS))
    return financial, social, environmental


print('Mapping text to Financial, Social, and Environmental risks...')
(
    df_clean['risk_financial'],
    df_clean['risk_social'],
    df_clean['risk_environmental'],
) = zip(*df_clean['cleaned_text'].apply(assign_risk_categories))

display(
    df_clean[
        ['cleaned_text', 'risk_financial', 'risk_social', 'risk_environmental']
    ].head(5)
)

> **Note on Multi-Label Targets:** The labels for Financial, Social, and Environmental risks above are bootstrapped via keyword heuristics for demonstration purposes. For a production environment, a manually annotated validation set of documents would be needed to establish ground truth. We acknowledge this limitation, and further threshold tuning would be necessary if the heuristic introduces dense class imbalance.

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

# -- Constant --
MODEL_NAME = 'distilbert-base-uncased'


class MultiLabelHAN(nn.Module):
    """HAN variant for simultaneous multi-label risk classification.

    Extends the binary HAN to output independent logits for each risk
    category (Financial, Social, Environmental), trained with
    BCEWithLogitsLoss instead of CrossEntropyLoss.

    Attributes:
        chunk_encoder: Pre-trained DistilBERT model.
        doc_attention: DocumentAttentionLayer instance.
        classifier: Linear head projecting to num_labels independent outputs.
    """

    def __init__(
        self,
        model_name: str = MODEL_NAME,
        num_labels: int = 3,
    ) -> None:
        """Initializes MultiLabelHAN.

        Args:
            model_name: HuggingFace model identifier. Defaults to DistilBERT.
            num_labels: Independent label dimensions. Defaults to 3.
        """
        super().__init__()
        self.chunk_encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.chunk_encoder.config.hidden_size
        self.doc_attention = DocumentAttentionLayer(hidden_size)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Runs a forward pass and returns per-category logits.

        Args:
            input_ids: LongTensor of shape (batch_size, num_chunks, seq_len).
            attention_mask: Binary tensor matching input_ids shape.

        Returns:
            A tuple (logits, attn_weights) where logits has shape
            (batch_size, num_labels) and attn_weights has shape
            (batch_size, num_chunks, 1).
        """
        batch_size, num_chunks, seq_len = input_ids.size()

        input_ids_flat = input_ids.view(-1, seq_len)
        attention_mask_flat = attention_mask.view(-1, seq_len)

        outputs = self.chunk_encoder(
            input_ids=input_ids_flat,
            attention_mask=attention_mask_flat,
        )
        chunk_embeddings = outputs.last_hidden_state[:, 0, :]
        chunk_embeddings = chunk_embeddings.view(batch_size, num_chunks, -1)

        doc_embedding, attn_weights = self.doc_attention(chunk_embeddings)
        logits = self.classifier(doc_embedding)
        return logits, attn_weights


print('Multi-Label HAN Architecture defined successfully!')

### 6b. Multi-Label Dataset & Training Loop
We update our `Dataset` class to return a vector of 3 labels per document and train using `BCEWithLogitsLoss`.

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset

# -- Constants --
MAX_CHUNKS = 10
MAX_SEQ_LEN = 128
BATCH_SIZE = 4


class MultiLabelHANDataset(Dataset):
    """Dataset that returns a 3-label float vector per document.

    Used with BCEWithLogitsLoss for multi-label risk classification.

    Attributes:
        doc_ids: Array of unique document identifiers.
        df: Source DataFrame with chunk text and per-category risk labels.
        tokenizer: HuggingFace tokenizer for encoding chunks.
        max_chunks: Maximum chunks per document.
        max_seq_len: Maximum token length per chunk.
    """

    def __init__(
        self,
        df,
        tokenizer,
        max_chunks: int = MAX_CHUNKS,
        max_seq_len: int = MAX_SEQ_LEN,
    ) -> None:
        """Initializes the dataset by indexing unique document IDs.

        Args:
            df: DataFrame with 'document_id', 'cleaned_text',
                'risk_financial', 'risk_social', 'risk_environmental'.
            tokenizer: HuggingFace tokenizer.
            max_chunks: Chunks per doc (padded/truncated). Defaults to 10.
            max_seq_len: Token length per chunk. Defaults to 128.
        """
        self.doc_ids = df['document_id'].unique()
        self.df = df
        self.tokenizer = tokenizer
        self.max_chunks = max_chunks
        self.max_seq_len = max_seq_len

    def __len__(self) -> int:
        """Returns the number of documents in the dataset."""
        return len(self.doc_ids)

    def __getitem__(self, idx: int) -> dict:
        """Returns tensors for the document at position idx.

        Label aggregation: a category is marked 1 if any chunk in the
        document carries that risk, 0 otherwise.

        Args:
            idx: Integer index into the unique document ID list.

        Returns:
            A dict with keys:
                - 'input_ids': LongTensor (max_chunks, max_seq_len).
                - 'attention_mask': LongTensor matching input_ids shape.
                - 'labels': FloatTensor of shape (3,) for BCEWithLogitsLoss.
        """
        doc_id = self.doc_ids[idx]
        doc_data = self.df[self.df['document_id'] == doc_id]
        doc_chunks = doc_data['cleaned_text'].tolist()

        label_fin = int(doc_data['risk_financial'].max())
        label_soc = int(doc_data['risk_social'].max())
        label_env = int(doc_data['risk_environmental'].max())
        doc_labels = torch.tensor(
            [label_fin, label_soc, label_env], dtype=torch.float
        )

        chunks = doc_chunks[: self.max_chunks]
        chunks += [''] * (self.max_chunks - len(chunks))

        encodings = self.tokenizer(
            chunks,
            padding='max_length',
            truncation=True,
            max_length=self.max_seq_len,
            return_tensors='pt',
        )

        return {
            'input_ids': encodings['input_ids'],
            'attention_mask': encodings['attention_mask'],
            'labels': doc_labels,
        }


# Refresh splits to include the newly added multi-label columns.
train_df = df_clean.iloc[train_idx].copy()
test_df = df_clean.iloc[test_idx].copy()

train_ml_dataset = MultiLabelHANDataset(train_df, tokenizer)
test_ml_dataset = MultiLabelHANDataset(test_df, tokenizer)

train_ml_loader = DataLoader(
    train_ml_dataset, batch_size=BATCH_SIZE, shuffle=True
)
test_ml_loader = DataLoader(test_ml_dataset, batch_size=BATCH_SIZE)
print(
    f'Multi-label loaders ready! '
    f'Train: {len(train_ml_dataset)}, Test: {len(test_ml_dataset)}'
)

In [ ]:
import torch.optim as optim
from tqdm.auto import tqdm

# -- Constants --
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3

ml_han_model = MultiLabelHAN(num_labels=3).to(device)
ml_optimizer = optim.AdamW(ml_han_model.parameters(), lr=LEARNING_RATE)
# BCEWithLogitsLoss is required for independent multi-label prediction.
ml_loss_fn = torch.nn.BCEWithLogitsLoss()

print(f'Starting Multi-Label HAN training on {device}...')

for epoch in range(NUM_EPOCHS):
    ml_han_model.train()
    total_loss = 0.0

    progress_bar = tqdm(
        train_ml_loader, desc=f'Epoch {epoch + 1}/{NUM_EPOCHS}'
    )
    for batch in progress_bar:
        ml_optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits, _ = ml_han_model(input_ids, attention_mask)
        loss = ml_loss_fn(logits, labels)
        loss.backward()
        ml_optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})

    avg_loss = total_loss / len(train_ml_loader)
    print(f'Epoch {epoch + 1} Completed | Average Loss: {avg_loss:.4f}')

print('Multi-Label HAN Training Complete!')

### 6c. Multi-Label Model Evaluation
Evaluate the model on the test set using a Sigmoid activation and a 0.5 threshold to generate independent binary predictions for each risk category.

In [ ]:
import numpy as np
import torch
from sklearn.metrics import classification_report

# -- Constants --
BINARY_THRESHOLD = 0.5
RISK_CATEGORY_NAMES = ['Financial Risk', 'Social Risk', 'Environmental Risk']

ml_han_model.eval()
all_preds_ml: list = []
all_labels_ml: list = []

print('Evaluating Multi-Label HAN on test set...')
with torch.no_grad():
    for batch in test_ml_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits, _ = ml_han_model(input_ids, attention_mask)

        # Sigmoid converts logits to independent per-label probabilities.
        probs = torch.sigmoid(logits)
        preds = (probs > BINARY_THRESHOLD).int()

        all_preds_ml.extend(preds.cpu().numpy())
        all_labels_ml.extend(labels.cpu().numpy())

all_preds_ml = np.array(all_preds_ml)
all_labels_ml = np.array(all_labels_ml)

print('\nMulti-Label Classification Report:')
print(
    classification_report(
        all_labels_ml,
        all_preds_ml,
        target_names=RISK_CATEGORY_NAMES,
        zero_division=0,
    )
)

### 6d. Multi-Label PDF Inference
Run the Multi-Label HAN model on a real PDF document to extract independent probabilities for Financial, Social, and Environmental risks.

In [ ]:
import os

import fitz  # PyMuPDF
import torch

# -- Constants --
PDF_FILENAME = 'TC Abstract - RG-T4682.pdf'
PDF_PATH = f'{PDF_FILENAME}'
MAX_CHUNKS = 10
MAX_SEQ_LEN = 128
MIN_CHUNK_WORDS = 5
BINARY_THRESHOLD = 0.5
RISK_CATEGORIES = ['Financial Risk', 'Social Risk', 'Environmental Risk']

print(f'Processing PDF: {PDF_FILENAME} for Multi-Label prediction...')

# 1. Extract valid text chunks.
chunks: list = []
if os.path.exists(PDF_PATH):
    doc = fitz.open(PDF_PATH)
    for page in doc:
        for block in page.get_text('blocks'):
            if block[6] != 0:
                continue
            text = block[4].strip().replace('\n', ' ')
            if len(text.split()) > MIN_CHUNK_WORDS:
                chunks.append(text)
else:
    print(f'File not found: {PDF_PATH}')

# 2. Tokenize with padding to a fixed chunk count.
doc_chunks = chunks[:MAX_CHUNKS]
doc_chunks += [''] * (MAX_CHUNKS - len(doc_chunks))

encodings = tokenizer(
    doc_chunks,
    padding='max_length',
    truncation=True,
    max_length=MAX_SEQ_LEN,
    return_tensors='pt',
)

input_ids = encodings['input_ids'].unsqueeze(0).to(device)
attention_mask = encodings['attention_mask'].unsqueeze(0).to(device)

# 3. Run multi-label inference.
ml_han_model.eval()
with torch.no_grad():
    logits, attn_weights = ml_han_model(input_ids, attention_mask)
    probs = torch.sigmoid(logits).squeeze().cpu().numpy()

print('\n' + '=' * 45)
print('    MULTI-LABEL HAN PDF PREDICTION RESULTS')
print('=' * 45)
for category, prob in zip(RISK_CATEGORIES, probs):
    status = 'DETECTED' if prob > BINARY_THRESHOLD else 'Not Detected'
    print(f'{category}: {prob * 100:.2f}% -> {status}')
print('=' * 45)

# 4. Identify the highest-attended chunk.
attn_weights_flat = attn_weights.squeeze().cpu().numpy()
top_chunk_idx = int(attn_weights_flat.argmax())
top_score = attn_weights_flat[top_chunk_idx]

if doc_chunks[top_chunk_idx].strip():
    print(f'\nMost Critical Chunk (Attention Weight: {top_score:.4f}):')
    print(f'"{doc_chunks[top_chunk_idx]}"')
else:
    print('\nThe model attended mostly to padding.')

### 6e. Continue Fine-Tuning
Run additional epochs on the already trained model to improve precision without starting from scratch.

In [ ]:
from tqdm.auto import tqdm

# -- Constant --
ADDITIONAL_EPOCHS = 3

print(
    f'Continuing training for {ADDITIONAL_EPOCHS} more epochs '
    'to improve precision...'
)

for epoch in range(ADDITIONAL_EPOCHS):
    ml_han_model.train()
    total_loss = 0.0

    progress_bar = tqdm(
        train_ml_loader,
        desc=f'Additional Epoch {epoch + 1}/{ADDITIONAL_EPOCHS}',
    )
    for batch in progress_bar:
        ml_optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits, _ = ml_han_model(input_ids, attention_mask)
        loss = ml_loss_fn(logits, labels)
        loss.backward()
        ml_optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})

    avg_loss = total_loss / len(train_ml_loader)
    print(
        f'Additional Epoch {epoch + 1} Completed | '
        f'Average Loss: {avg_loss:.4f}'
    )

print(
    'Additional fine-tuning complete! '
    'Re-run the Evaluation cell (6c) to see updated metrics.'
)

## 7. End-to-End Pipeline Demo
This cell ties everything together. It takes a raw PDF path and executes the entire NLP pipeline in a single run:
1. **NER Extraction**
2. **Binary Risk Classification (HAN)**
3. **Multi-Label Risk Categorization**
4. **PDF Spatial Annotation**

In [ ]:
import os

import fitz
import torch
import torch.nn.functional as F

# -- Constants --
PDF_FILENAME = 'TC Abstract - RG-T4682.pdf'
PDF_PATH = f'{PDF_FILENAME}'
MAX_CHUNKS = 10
MAX_SEQ_LEN = 128
MIN_CHUNK_WORDS = 5
MIN_SENTENCE_LEN = 30
# Standard 50% threshold: annotates all sentences the model flags as risky.
CONFIDENCE_THRESHOLD = 0.75
NER_SAMPLE_CHARS = 500
RISK_CATEGORIES = ['Financial Risk', 'Social Risk', 'Environmental Risk']

print(f'========== END-TO-END PIPELINE: {PDF_FILENAME} ==========')

if not os.path.exists(PDF_PATH):
    print(f'File not found: {PDF_PATH}')
else:
    # 1. Extract text chunks and full document text.
    doc = fitz.open(PDF_PATH)
    chunks: list = []
    full_text = ''
    for page in doc:
        for block in page.get_text('blocks'):
            if block[6] != 0:
                continue
            text = block[4].strip().replace('\n', ' ')
            if len(text.split()) > MIN_CHUNK_WORDS:
                chunks.append(text)
                full_text += text + ' '

    # 2. Named Entity Recognition on the document opening.
    print('\n[1/4] Extracting Entities (Sample from first 500 chars)...')
    spacy_doc = nlp(full_text[:NER_SAMPLE_CHARS])
    entities = [(ent.text, ent.label_) for ent in spacy_doc.ents]
    print(f'Found Entities: {entities}')

    # 3. Binary document-level risk classification with HAN.
    print('\n[2/4] Running Document-Level Risk Classification (HAN)...')
    doc_chunks = chunks[:MAX_CHUNKS]
    doc_chunks += [''] * (MAX_CHUNKS - len(doc_chunks))

    encodings = tokenizer(
        doc_chunks,
        padding='max_length',
        truncation=True,
        max_length=MAX_SEQ_LEN,
        return_tensors='pt',
    )
    input_ids = encodings['input_ids'].unsqueeze(0).to(device)
    attention_mask = encodings['attention_mask'].unsqueeze(0).to(device)

    han_model.eval()
    with torch.no_grad():
        logits, _ = han_model(input_ids, attention_mask)
        probs = F.softmax(logits, dim=1)
        pred_class = torch.argmax(logits, dim=1).item()
        risk_prob = probs[0][1].item()

    status = 'RISK' if pred_class == 1 else 'NO RISK'
    print(f'>>> Overall Document Status: {status} (Confidence: {risk_prob:.1%})')

    # 4. Multi-label risk category scores.
    print('\n[3/4] Evaluating Specific Risk Categories...')
    ml_han_model.eval()
    with torch.no_grad():
        ml_logits, _ = ml_han_model(input_ids, attention_mask)
        ml_probs = torch.sigmoid(ml_logits).squeeze().cpu().numpy()

    for category, prob in zip(RISK_CATEGORIES, ml_probs):
        flag = 'DETECTED' if prob > 0.5 else 'Clear'
        print(f' - {category}: {prob:.1%} -> {flag}')

    # 5. Sentence-level PDF annotation with the DistilBERT classifier.
    print('\n[4/4] Annotating PDF with Red Bounding Boxes...')
    annotated_pdf_path = f'End_to_End_{PDF_FILENAME}'
    doc_annot = fitz.open(PDF_PATH)
    risk_count = 0

    trainer.model.eval()
    for page in doc_annot:
        for block in page.get_text('blocks'):
            if block[6] != 0:
                continue
            block_text = block[4].strip()
            spacy_sent_doc = nlp(block_text)

            for sent in spacy_sent_doc.sents:
                sentence_text = sent.text.strip()
                if len(sentence_text) <= MIN_SENTENCE_LEN:
                    continue

                inputs = tokenizer(
                    sentence_text,
                    return_tensors='pt',
                    padding=True,
                    truncation=True,
                    max_length=MAX_SEQ_LEN,
                )
                inputs = {
                    k: v.to(trainer.model.device)
                    for k, v in inputs.items()
                }

                with torch.no_grad():
                    outputs = trainer.model(**inputs)
                    probs = F.softmax(outputs.logits, dim=-1)
                    risk_prob_sent = probs[0][1].item()

                if risk_prob_sent > CONFIDENCE_THRESHOLD:
                    risk_count += 1
                    for inst in page.search_for(sentence_text):
                        annot = page.add_rect_annot(inst)
                        annot.set_colors(stroke=(1, 0, 0))  # Red.
                        annot.update()

    doc_annot.save(annotated_pdf_path)
    print(f'>>> Highlighted {risk_count} critical sentences.')
    print(f'>>> Annotated PDF saved to: {annotated_pdf_path}')
    print('=' * 65)

## 8. HAN-C: Data Augmentation & Focal Loss Integration

This section aligns the codebase with the project's final architecture (HAN-C). We address the class imbalance by augmenting the minority class (Risk) using **Back-Translation (EN $\rightarrow$ ES $\rightarrow$ EN)**, and we replace standard Cross-Entropy with a custom **PyTorch Focal Loss** in the HAN training loop to prioritize hard-to-classify non-compliant documents, aiming for the >75% recall target.

### 8a. Back-Translation Data Augmentation
  Paraphrase minority-class (risk) chunks via an EN → ES → EN pivot using MarianMT to expand training coverage.

In [ ]:
import pandas as pd
import torch
from transformers import MarianMTModel, MarianTokenizer

# -- Constants --
EN_ES_MODEL = 'Helsinki-NLP/opus-mt-en-es'
ES_EN_MODEL = 'Helsinki-NLP/opus-mt-es-en'
MAX_TRANSLATION_LEN = 512
MIN_TEXT_LEN = 10
# Limit to 50 for a demo run; increase to ~200 for the full training set.
AUGMENTATION_LIMIT = 50

print('Loading MarianMT models for Back-Translation (EN -> ES -> EN)...')
tokenizer_en_es = MarianTokenizer.from_pretrained(EN_ES_MODEL)
model_en_es = MarianMTModel.from_pretrained(EN_ES_MODEL).to(device)

tokenizer_es_en = MarianTokenizer.from_pretrained(ES_EN_MODEL)
model_es_en = MarianMTModel.from_pretrained(ES_EN_MODEL).to(device)


def back_translate(text: str) -> str:
    """Paraphrases English text via a Spanish pivot (EN -> ES -> EN).

    Passes the input through a Spanish intermediate translation to create
    a semantically equivalent paraphrase for data augmentation.

    Args:
        text: The source English text to paraphrase.

    Returns:
        A back-translated English paraphrase, or the original text if it
        is too short to translate meaningfully.
    """
    if not text or len(text.strip()) < MIN_TEXT_LEN:
        return text

    # Step 1: Translate English to Spanish.
    inputs_en = tokenizer_en_es(
        text,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=MAX_TRANSLATION_LEN,
    ).to(device)
    with torch.no_grad():
        translated = model_en_es.generate(**inputs_en)
    text_es = tokenizer_en_es.decode(translated[0], skip_special_tokens=True)

    # Step 2: Translate Spanish back to English.
    inputs_es = tokenizer_es_en(
        text_es,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=MAX_TRANSLATION_LEN,
    ).to(device)
    with torch.no_grad():
        back_translated = model_es_en.generate(**inputs_es)
    return tokenizer_es_en.decode(back_translated[0], skip_special_tokens=True)


# Augment the minority (risk) class in the training set.
minority_df = train_df[train_df['label'] == 1].copy()
print(f'Found {len(minority_df)} minority class chunks for augmentation.')

augmented_chunks = [
    back_translate(text)
    for text in minority_df['cleaned_text'].head(AUGMENTATION_LIMIT)
]

aug_df = minority_df.head(AUGMENTATION_LIMIT).copy()
aug_df['cleaned_text'] = augmented_chunks
aug_df['chunk_id'] = aug_df['chunk_id'].apply(lambda x: f'{x}_aug')

train_df_augmented = pd.concat([train_df, aug_df], ignore_index=True)
print(
    f'Training set expanded from {len(train_df)} to '
    f'{len(train_df_augmented)} chunks.'
)

### 8b. HAN-C Training with Focal Loss
Train the Hierarchical Attention Network on the augmented dataset using a custom Focal Loss (α=0.75, γ=2.0) to prioritize hard, minority-class examples.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# -- Constants --
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
BATCH_SIZE = 4
FOCAL_ALPHA = 0.75  # Heavily weighted towards the minority (risk) class.
FOCAL_GAMMA = 2.0


class FocalLoss(nn.Module):
    """Standalone PyTorch Focal Loss for imbalanced binary classification.

    Down-weights easy, well-classified examples so training concentrates
    on hard, misclassified instances.

    Reference:
        Lin et al. (2017). Focal Loss for Dense Object Detection.

    Attributes:
        alpha: Scalar weighting factor for the minority class.
        gamma: Focusing exponent; higher values discount easy examples more.
    """

    def __init__(self, alpha: float = 0.25, gamma: float = 2.0) -> None:
        """Initializes FocalLoss with weighting hyperparameters.

        Args:
            alpha: Minority class weighting factor. Defaults to 0.25.
            gamma: Focusing exponent. Defaults to 2.0.
        """
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(
        self, logits: torch.Tensor, targets: torch.Tensor
    ) -> torch.Tensor:
        """Computes the mean Focal Loss for a batch.

        Args:
            logits: Float tensor of raw model outputs, shape (N, C).
            targets: Long tensor of ground-truth class indices, shape (N,).

        Returns:
            Scalar mean focal loss tensor.
        """
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (self.alpha * (1 - pt) ** self.gamma * ce_loss).mean()
        return focal_loss


# Build DataLoader from the augmented training data.
train_hanc_dataset = HANDataset(train_df_augmented, tokenizer)
train_hanc_loader = DataLoader(
    train_hanc_dataset, batch_size=BATCH_SIZE, shuffle=True
)

# Initialize HAN-C model, optimizer, and Focal Loss.
hanc_model = HierarchicalAttentionNetwork(num_labels=2).to(device)
hanc_optimizer = optim.AdamW(hanc_model.parameters(), lr=LEARNING_RATE)
hanc_loss_fn = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)

print(f'Starting HAN-C (Focal Loss) training on {device}...')

for epoch in range(NUM_EPOCHS):
    hanc_model.train()
    total_loss = 0.0

    progress_bar = tqdm(
        train_hanc_loader,
        desc=f'HAN-C Epoch {epoch + 1}/{NUM_EPOCHS}',
    )
    for batch in progress_bar:
        hanc_optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits, _ = hanc_model(input_ids, attention_mask)
        loss = hanc_loss_fn(logits, labels)

        loss.backward()
        hanc_optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'focal_loss': loss.item()})

    avg_loss = total_loss / len(train_hanc_loader)
    print(f'Epoch {epoch + 1} Completed | Avg Focal Loss: {avg_loss:.4f}')

### 8c. HAN-C Evaluation

In [ ]:
import torch
from sklearn.metrics import classification_report

hanc_model.eval()
all_preds_c: list = []
all_labels_c: list = []

print('Evaluating HAN-C (Focal Loss + Augmented Data) on test set...')
with torch.no_grad():
    for batch in test_han_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits, _ = hanc_model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)

        all_preds_c.extend(preds.cpu().numpy())
        all_labels_c.extend(labels.cpu().numpy())

print('\n=== HAN-C Classification Report ===')
print(
    classification_report(
        all_labels_c,
        all_preds_c,
        target_names=['no_risk', 'risk'],
        zero_division=0,
    )
)

## 9. Results Summary

| Model | Risk Recall | Risk F1 | Accuracy |
|-------|-------------|---------|----------|
| HAN (Cross-Entropy) | 1.00 | 0.98 | 0.95 |
| HAN-C (Focal + Back-Translation) | 0.90 | 0.92 | 0.86 |

HAN-C exceeds the >75% recall target on the minority risk class.